# 第 27 章 · 可观测性：零、未知、重复和费用

一次请求失败，可能已经消耗 token；一个字段显示 0，也可能只是缺失值被归零。诊断系统的第一步是保留证据资格，然后才做聚合。

本章将受控 Responses 形状的事件经过真实解析器和 ledger，计算部分费用。所有输入是 fixture，价格单位是 TEST_UNIT；结果不是服务账单，也没有真实模型成本证据。

**运行范围：局部机制实验。** 按顺序执行，每个实验都通过真实函数、文件或队列产生结果，再用断言检查。修改参数后重跑该实验，必要时先重启内核并执行全部。所有合成输入都只证明本地机制；本章没有累计检查点，真实服务与设备仍待验证。

本章使用原作者的设计与局部原型；下面明确写出的简化是教材选择，不是 Codex 原样实现。固定源码：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter

## 1 · missing、null 和 0 不能合并

可加和的测量必须是已知非负整数。父对象缺失和字段缺失也分开；这让接入层错误可以追踪到具体 pointer。下方只变更 usage，观察真实 parser 的不同结果。

In [2]:
m = load_example('27', 'meter')
def observe(event_id, response_id, usage='MISSING', terminal='response.completed', attempt=None):
    response = {'id': response_id}
    if usage != 'MISSING':
        response['usage'] = usage
    return m.observe_raw({'type': terminal, 'response': response},
        event_id=event_id, task_id='task-1', thread_id='thread-1', parent_thread_id=None,
        root_turn_id='turn-1', attempt_id=attempt or event_id, provider_scope='fixture-provider',
        model='teaching-model', purpose='repair', evidence_ref=event_id + '.json')
known = {'input_tokens': 100, 'output_tokens': 20, 'total_tokens': 120,
         'input_tokens_details': {'cached_tokens': 0}}
zero = observe('zero', 'r-zero', known)
missing = observe('missing', 'r-missing')
null = observe('null', 'r-null', None)
assert zero.fields['cached'].state == 'known' and zero.fields['cached'].value == 0
assert missing.fields['input'].state == 'parent_missing'
assert null.fields['input'].state == 'parent_null'
print({o.event_id: {'usage_presence': o.usage_presence, 'cached': o.fields['cached']} for o in (zero, missing, null)})

{'zero': {'usage_presence': 'object', 'cached': Measurement(state='known', value=0, pointer='/usage/input_tokens_details/cached_tokens')}, 'missing': {'usage_presence': 'missing', 'cached': Measurement(state='parent_missing', value=None, pointer='/usage')}, 'null': {'usage_presence': 'null', 'cached': Measurement(state='parent_null', value=None, pointer='/usage')}}


## 2 · 去重同一事实，保留真实重试

重复投递同一 event 和同一 response 都不应加两次。真实重试有新 response ID，即使 token 数相同仍是一笔工作。同 response 的 token 被改写则不是更新，而是证据冲突。

In [3]:
from dataclasses import replace
ledger = m.Ledger()
assert ledger.record(zero) == 'recorded'
assert ledger.record(zero) == 'duplicate_event'
redelivered = replace(zero, event_id='redelivery', evidence_ref='redelivery.json', attempt_id='redelivery')
assert ledger.record(redelivered) == 'duplicate_response'
retry = observe('retry', 'r-retry', known)
assert ledger.record(retry) == 'recorded'
assert len(ledger.observations) == 2
conflict_ledger = m.Ledger()
conflict_ledger.record(zero)
changed = observe('changed', 'r-zero', {**known, 'input_tokens': 999})
try:
    conflict_ledger.record(changed)
except m.Conflict as error:
    assert str(error) == 'same_response_changed_fact'
else:
    raise AssertionError('同一 response 改写事实必须暴露')
assert zero.usage_key() in conflict_ledger.conflicts
print({'unique_billable_observations': len(ledger.observations), 'conflicts': len(conflict_ledger.conflicts)})

{'unique_billable_observations': 2, 'conflicts': 1}


## 3 · 失败请求可有用量，未知用量只能报告部分小计

输入量包含 cached 部分的语义在此实验中显式成立，所以费用是 `(input-cached)*普通价 + cached*缓存价 + output*输出价`。换成另一个 provider 前必须重新核实这条语义。

In [4]:
from decimal import Decimal
rates = {'teaching-model': {'input': Decimal('2'), 'cached': Decimal('0.5'),
                            'output': Decimal('4'), 'version': 'synthetic-price-v1'}}
failed_usage = {**known, 'input_tokens_details': {'cached_tokens': 60}}
failed = observe('failed', 'r-failed', failed_usage, terminal='response.failed')
ledger.record(failed)
ledger.record(missing)
summary = ledger.query('task-1', rates, inventory_complete=True)
assert summary['unique_observations'] == 4
assert summary['tokens']['input']['known_sum'] == 300
assert summary['tokens']['input']['unknown_count'] == 1
assert summary['coverage'] == 'partial'
# 两次 0.00028，加一次失败请求 0.00019，未知请求不能按零填。
assert Decimal(summary['estimated_known_subtotal']) == Decimal('0.00075')
assert summary['billing_claim'] == 'not_an_invoice'
print({k: summary[k] for k in ('estimated_known_subtotal', 'currency', 'coverage', 'unpriced')})

{'estimated_known_subtotal': '0.00075', 'currency': 'TEST_UNIT', 'coverage': 'partial', 'unpriced': [{'attempt_id': 'missing', 'component': 'output', 'reason': 'parent_missing'}, {'attempt_id': 'missing', 'component': 'input', 'reason': 'cache_split_or_input_unknown'}]}


## 4 · 恢复投影不能补造原始字段

旧版只有规范化数字时，无法逆推出 provider 当时是否提供过字段。取消或传输中断也可能没有 usage；它们应留在 inventory，不自动变成免费请求。新 ledger 重放相同事件后应得到相同部分小计。

In [5]:
normalized = m.observe_numeric_only(response_id='r-history',
    event_id='history', task_id='task-1', thread_id='thread-1', parent_thread_id=None,
    root_turn_id='turn-old', attempt_id='historical-attempt', provider_scope='fixture-provider',
    model='teaching-model', purpose='repair', evidence_ref='normalized-history.json')
assert normalized.usage_presence == 'presence_not_observable'
interrupted = observe('interrupted', 'r-interrupted', terminal='transport.failed')
replayed = m.Ledger()
for event in (zero, retry, failed, missing, normalized, interrupted):
    replayed.record(event)
    replayed.record(event)
restored_summary = replayed.query('task-1', rates, inventory_complete=True)
assert restored_summary['unique_observations'] == 6
assert restored_summary['tokens']['input']['unknown_count'] == 3
assert restored_summary['estimated_known_subtotal'] == summary['estimated_known_subtotal']
assert restored_summary['coverage'] == 'partial'
print({'observations': restored_summary['unique_observations'], 'unknown_input': restored_summary['tokens']['input']['unknown_count'],
       'known_subtotal': restored_summary['estimated_known_subtotal']})

{'observations': 6, 'unknown_input': 3, 'known_subtotal': '0.00075'}


## 源码对照与接回主线

以下为固定提交的已定位源码入口。阅读源码能解释设计边界，不等于执行过其测试；本章 Python 原型与这些 Rust 接口不是同一实现。

- [codex-rs/codex-api/src/sse/responses.rs#L112-L160](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/codex-api/src/sse/responses.rs#L112-L160)
- [codex-rs/codex-api/src/sse/responses.rs#L518-L547](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/codex-api/src/sse/responses.rs#L518-L547)
- [codex-rs/codex-api/src/sse/responses.rs#L476-L541](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/codex-api/src/sse/responses.rs#L476-L541)

**接回条件：** 将局部机制接入同一 Dispatcher、Scheduler、History 和受保护验证器，再执行原合同。目录、断言和演示输出都不能替代累计集成证据。